# NOTEBOOK CHI TIẾT — PHẦN V: BOOTSTRAP CEPH CLUSTER

**Mục tiêu:** dựng Ceph cluster bare metal trên 3 server vật lý để tích hợp OpenStack/Kolla-Ansible.

- **Server A / stack1:** `192.168.150.9`
- **Server B / stack2:** `192.168.150.10`
- **Server C / stack3:** `192.168.150.11`

Ceph chạy **bare metal**, không chạy trong Controller VM.

## IP dùng trong phần Ceph

| Thành phần | Server A | Server B | Server C | Mục đích |
|---|---:|---:|---:|---|
| br-storage | 10.0.2.21 | 10.0.2.22 | 10.0.2.23 | Ceph public/client network |
| br-cluster | 10.0.3.21 | 10.0.3.22 | 10.0.3.23 | Ceph OSD replication/recovery |
| br-mgmt | 10.0.0.21 | 10.0.0.22 | 10.0.0.23 | SSH/Ansible/Kolla management |

> Quy tắc quan trọng: **MON listen trên br-storage `10.0.2.x`**, còn **br-cluster `10.0.3.x` chỉ dùng cho replication/backfill/recovery giữa OSD**.

# 0. Kiểm tra trước khi bắt đầu

Chạy trên **cả 3 server** để chắc chắn network đã có đủ `br-storage` và `br-cluster`.

In [ ]:
hostname
ip -br addr show br-storage
ip -br addr show br-cluster
ip -br addr show br-mgmt

Kỳ vọng:

- Server A có:
  - `br-storage 10.0.2.21/24`
  - `br-cluster 10.0.3.21/24`
- Server B có:
  - `br-storage 10.0.2.22/24`
  - `br-cluster 10.0.3.22/24`
- Server C có:
  - `br-storage 10.0.2.23/24`
  - `br-cluster 10.0.3.23/24`

Trên **Server A**, test ping sang B/C:

In [ ]:
ping -c 3 -I br-storage 10.0.2.22
ping -c 3 -I br-storage 10.0.2.23

ping -c 3 -I br-cluster 10.0.3.22
ping -c 3 -I br-cluster 10.0.3.23

# BƯỚC 20 — Chuẩn bị và cài Ceph tools

Thực hiện trên **cả 3 server A/B/C**.

In [ ]:
sudo apt update
sudo apt install -y ceph-common lvm2 gdisk smartmontools chrony curl wget

## 20.1. Đồng bộ thời gian

Ceph rất nhạy với lệch giờ. Nếu clock skew, MON có thể warning hoặc cluster không ổn định.

In [ ]:
sudo systemctl enable --now chrony
chronyc sources
chronyc tracking

Kỳ vọng:

- `chronyc tracking` có offset nhỏ.
- Nếu lệch quá lớn, cần chờ chrony đồng bộ hoặc kiểm tra NTP/DNS/Internet.

## 20.2. Cấu hình `/etc/hosts`

Chạy trên **cả 3 server**. Dùng IP `br-storage` vì đây là public/client network của Ceph.

In [ ]:
sudo cp /etc/hosts /etc/hosts.bak.$(date +%F-%H%M%S)

sudo tee -a /etc/hosts << EOF
10.0.2.21  server-a
10.0.2.22  server-b
10.0.2.23  server-c
EOF

cat /etc/hosts | tail -n 10

## 20.3. SSH passwordless từ Server A sang B/C qua br-storage

Chỉ chạy trên **Server A**.

In [ ]:
ssh-keygen -t ed25519 -f ~/.ssh/ceph_key -N "" -C "ceph-deploy-key"
ssh-copy-id -i ~/.ssh/ceph_key.pub ubuntu@10.0.2.22
ssh-copy-id -i ~/.ssh/ceph_key.pub ubuntu@10.0.2.23

Test SSH từ Server A sang B/C:

In [ ]:
ssh -i ~/.ssh/ceph_key ubuntu@10.0.2.22 "hostname && ip -br addr show br-storage"
ssh -i ~/.ssh/ceph_key ubuntu@10.0.2.23 "hostname && ip -br addr show br-storage"

# BƯỚC 21 — Xác định và chuẩn bị OSD disk

Thực hiện trên **cả 3 server**.

⚠️ Cực kỳ cẩn thận: chọn sai disk là mất dữ liệu OS hoặc LVM VM.

In [ ]:
lsblk -d -o NAME,SIZE,TYPE,MOUNTPOINT
lsblk -f
sudo pvs
sudo vgs
sudo lvs

## 21.1. Quy tắc chọn disk OSD

Disk dùng làm OSD phải thỏa:

- Không phải disk OS.
- Không có mountpoint `/`.
- Không thuộc `vg-lab`.
- Không chứa Controller VM.
- Nên là disk/partition riêng, ví dụ `/dev/sdc`.

Trong tài liệu mẫu dùng `/dev/sdc`. Nếu máy bạn dùng disk khác thì thay biến `OSD_DISK`.

In [ ]:
# Sửa dòng này theo disk thật của từng server
export OSD_DISK=/dev/sdc

echo "OSD_DISK=$OSD_DISK"
lsblk $OSD_DISK

## 21.2. Xóa sạch disk OSD

Chạy trên **cả 3 server** sau khi đã chắc chắn `OSD_DISK` đúng.

In [ ]:
sudo wipefs -a $OSD_DISK
sudo sgdisk --zap-all $OSD_DISK
sudo partprobe $OSD_DISK || true
lsblk $OSD_DISK

Kỳ vọng: disk không còn partition cũ.

# BƯỚC 22 — Bootstrap Ceph từ Server A

Chỉ chạy trên **Server A**.

- `--mon-ip 10.0.2.21`: MON đầu tiên listen trên br-storage.
- `--cluster-network 10.0.3.0/24`: OSD replication/recovery dùng br-cluster.

In [ ]:
cd ~
curl --silent --remote-name --location   https://github.com/ceph/ceph/raw/reef/src/cephadm/cephadm

chmod +x cephadm
sudo ./cephadm add-repo --release reef
sudo ./cephadm install

which cephadm
cephadm version || true

## 22.1. Bootstrap cluster

Chạy trên **Server A**.

In [ ]:
sudo cephadm bootstrap   --mon-ip 10.0.2.21   --cluster-network 10.0.3.0/24   --initial-dashboard-user admin   --initial-dashboard-password Admin@Ceph2025!

## 22.2. Kiểm tra sau bootstrap

Sau bootstrap, cluster thường mới có 1 MON nên có thể `HEALTH_WARN`, điều này bình thường.

In [ ]:
sudo ceph -s
sudo ceph mon stat
sudo ceph mgr stat
sudo ceph orch host ls

## 22.3. Kiểm tra Ceph network config

Xác nhận Ceph nhận đúng public/cluster network.

In [ ]:
sudo ceph config dump | grep -E "public_network|cluster_network" || true
sudo ceph config get mon public_network || true
sudo ceph config get mon cluster_network || true

# BƯỚC 23 — Thêm Server B/C và deploy MON/MGR/OSD

Chạy trên **Server A**.

## 23.1. Copy cephadm SSH key sang B/C

Cephadm dùng key `/etc/ceph/ceph.pub` để SSH quản lý node.

In [ ]:
sudo ssh-copy-id -f -i /etc/ceph/ceph.pub root@10.0.2.22
sudo ssh-copy-id -f -i /etc/ceph/ceph.pub root@10.0.2.23

Nếu lỗi root SSH, kiểm tra trên Server B/C:

```bash
sudo passwd root
sudo nano /etc/ssh/sshd_config
# PermitRootLogin yes
sudo systemctl restart ssh
```

Sau đó chạy lại `ssh-copy-id`.

## 23.2. Add host vào Ceph cluster

Dùng IP `br-storage`.

In [ ]:
sudo ceph orch host add server-b 10.0.2.22
sudo ceph orch host add server-c 10.0.2.23

sudo ceph orch host ls

Kỳ vọng thấy 3 host: `server-a`, `server-b`, `server-c`.

## 23.3. Deploy 3 MON

Deploy 3 MON để có quorum 3 node.

In [ ]:
sudo ceph orch apply mon 3
sleep 30
sudo ceph mon stat
sudo ceph quorum_status --format json-pretty

Kỳ vọng: quorum có 3 MON.

## 23.4. Deploy 2 MGR

MGR có thể active + standby.

In [ ]:
sudo ceph orch apply mgr 2
sleep 30
sudo ceph mgr stat
sudo ceph orch ps --daemon-type mgr

## 23.5. Kiểm tra disk available cho OSD

Chạy trên Server A để nhìn toàn cluster.

In [ ]:
sudo ceph orch device ls

## 23.6. Add OSD

Mẫu dùng `/dev/sdc`. Nếu disk thật khác thì sửa lại.

In [ ]:
sudo ceph orch daemon add osd server-a:/dev/sdc
sudo ceph orch daemon add osd server-b:/dev/sdc
sudo ceph orch daemon add osd server-c:/dev/sdc

## 23.7. Theo dõi OSD lên

Chờ vài phút.

In [ ]:
watch -n 5 'sudo ceph -s; echo; sudo ceph osd tree'

Thoát `watch`: nhấn `Ctrl + C`.

Sau đó kiểm tra lại:

In [ ]:
sudo ceph osd tree
sudo ceph osd stat
sudo ceph -s

Kỳ vọng:

- 3 OSD.
- Tất cả `up` và `in`.
- Cluster dần về `HEALTH_OK`.

# BƯỚC 24 — Tạo Pool và User cho OpenStack

Chạy trên **Server A**.

## 24.1. Tạo pool

Các pool:

- `images`: Glance image.
- `volumes`: Cinder volume.
- `vms`: Nova ephemeral disk.
- `backups`: Cinder backup.

In [ ]:
for pool in images volumes vms backups; do
  sudo ceph osd pool create $pool 32
  sudo ceph osd pool application enable $pool rbd
done

sudo ceph osd pool ls detail

## 24.2. Kiểm tra replica size

Với 3 OSD/3 server, replica size thường là 3.

In [ ]:
for pool in images volumes vms backups; do
  echo "===== $pool ====="
  sudo ceph osd pool get $pool size
  sudo ceph osd pool get $pool min_size
done

Nếu cần set rõ replica:

In [ ]:
for pool in images volumes vms backups; do
  sudo ceph osd pool set $pool size 3
  sudo ceph osd pool set $pool min_size 2
done

## 24.3. Tạo Ceph users/keyring cho OpenStack

Chạy trên Server A.

In [ ]:
sudo ceph auth get-or-create client.glance   mon "profile rbd"   osd "profile rbd pool=images"   > /etc/ceph/ceph.client.glance.keyring

sudo ceph auth get-or-create client.cinder   mon "profile rbd"   osd "profile rbd pool=volumes, profile rbd pool=vms"   > /etc/ceph/ceph.client.cinder.keyring

sudo ceph auth get-or-create client.nova   mon "profile rbd"   osd "profile rbd pool=vms, profile rbd pool=images"   > /etc/ceph/ceph.client.nova.keyring

sudo ceph auth get-or-create client.cinder-backup   mon "profile rbd"   osd "profile rbd pool=backups"   > /etc/ceph/ceph.client.cinder-backup.keyring

## 24.4. Kiểm tra keyring đã tạo

In [ ]:
sudo ls -l /etc/ceph/ceph.client.*.keyring
sudo ceph auth ls | egrep "client.glance|client.cinder|client.nova|client.cinder-backup"

## 24.5. Tạo `ceph.conf` sạch cho OpenStack

File này sẽ được copy vào Controller VM và thư mục Kolla config.

In [ ]:
FSID=$(sudo ceph fsid)

sudo tee /etc/ceph/ceph.conf << EOF
[global]
fsid = ${FSID}
mon_host = 10.0.2.21,10.0.2.22,10.0.2.23
auth_cluster_required = cephx
auth_service_required = cephx
auth_client_required = cephx
EOF

cat /etc/ceph/ceph.conf

Kiểm tra không có tab character:

In [ ]:
cat -A /etc/ceph/ceph.conf | grep -c "^I" || echo "Không có tab — OK"

## 24.6. Copy Ceph config về 3 Controller VM

Controller VM dùng:

- `10.0.0.11`
- `10.0.0.12`
- `10.0.0.13`

Lưu ý: các VM phải SSH được qua `br-mgmt`.

In [ ]:
for vm in 10.0.0.11 10.0.0.12 10.0.0.13; do
  echo "Copy Ceph config to $vm"
  sudo scp /etc/ceph/ceph.conf            /etc/ceph/ceph.client.*.keyring            ubuntu@$vm:/tmp/
done

## 24.7. Verify trên từng Controller VM

Chạy từ Server A:

In [ ]:
for vm in 10.0.0.11 10.0.0.12 10.0.0.13; do
  echo "===== $vm ====="
  ssh ubuntu@$vm "ls -l /tmp/ceph.conf /tmp/ceph.client.*.keyring"
done

# Kiểm tra cuối cùng trước khi sang Kolla-Ansible

Chạy trên Server A.

In [ ]:
sudo ceph -s
sudo ceph osd tree
sudo ceph orch host ls
sudo ceph orch ps
sudo ceph osd pool ls

# Checklist đạt yêu cầu

Trước khi chuyển sang phần Kolla-Ansible, cần đạt:

- `ceph -s` không có lỗi nghiêm trọng.
- Có 3 host: `server-a`, `server-b`, `server-c`.
- Có 3 MON quorum.
- Có 2 MGR hoặc ít nhất 1 MGR active.
- Có 3 OSD `up/in`.
- Có đủ pool: `images`, `volumes`, `vms`, `backups`.
- Có đủ keyring:
  - `client.glance`
  - `client.cinder`
  - `client.nova`
  - `client.cinder-backup`
- 3 Controller VM đã nhận file `/tmp/ceph.conf` và keyring.

# Debug nhanh các lỗi hay gặp

## Lỗi 1: `ceph -s` treo hoặc không connect được MON

```bash
sudo systemctl list-units | grep ceph | grep mon
sudo cephadm shell -- ceph -s
```

## Lỗi 2: Host B/C add không được

Kiểm tra SSH root từ Server A:

```bash
ssh root@10.0.2.22 hostname
ssh root@10.0.2.23 hostname
```

## Lỗi 3: OSD không lên

```bash
sudo ceph orch device ls
sudo ceph orch ps --daemon-type osd
sudo ceph health detail
```

## Lỗi 4: Clock skew

Trên cả 3 server:

```bash
chronyc tracking
chronyc sources
sudo systemctl restart chrony
```